In [387]:
import pandas as pd
import numpy as np
import plotly.express as px
import random
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression # Da/Ne
from sklearn.linear_model import LinearRegression # Brojevi
from sklearn.ensemble import RandomForestClassifier # Da/Ne
from sklearn.ensemble import RandomForestRegressor # Brojevi
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier # Da/Ne
from sklearn.tree import DecisionTreeRegressor # Brojevi
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from datetime import datetime, timedelta
import joblib

np.random.seed(42)
random.seed(42)

n_rows = 1200
proizvodi = {
    'Laptop ASUS ZenBook': ('Elektronika', 1100, 750),
    'HP Pavilion Gaming': ('Elektronika', 950, 680),
    'iPhone 15 Pro': ('Elektronika', 1200, 850),
    'Samsung S24 Ultra': ('Elektronika', 1300, 900),
    'Radni Sto Podesivi': ('Namestaj', 250, 120),
    'Kancelarijska Stolica': ('Namestaj', 150, 70),
    'Mehanicka Tastatura': ('Periferije', 80, 35),
    'Gaming Mis RGB': ('Periferije', 50, 20),
    'Dell Monitor 27': ('Elektronika', 300, 180),
    'Slusalice ANC': ('Periferije', 110, 50)
}

data = []
start_date = datetime(2025, 1, 1)

for i in range(n_rows):
    prod = random.choice(list(proizvodi.keys()))
    kat, cena, nabavna = proizvodi[prod]
    
    # Dodavanje varijacije u cenu i kolicinu
    cena_sa_popustom = round(cena * np.random.uniform(0.85, 1.05), 2)
    kolicina = random.randint(1, 5)
    
    # Generisanje datuma
    dan_offset = random.randint(0, 450)
    datum = start_date + timedelta(days=dan_offset)
    
    # Prljanje podataka (tekst u brojevima)
    cena_str = f"{cena_sa_popustom} RSD" if random.random() > 0.1 else cena_sa_popustom
    kolicina_val = float(kolicina) if random.random() > 0.05 else np.nan
    
    # Lokacija kupca
    grad = random.choice(['Subotica', 'Beograd', 'Novi Sad', 'Nis', 'Kragujevac', None])
    
    data.append([f"ORD_{10000+i}", datum.strftime('%Y-%m-%d'), prod, kat, kolicina_val, cena_str, nabavna, grad])

df = pd.DataFrame(data, columns=['ID_Porudzbine', 'Datum', 'Proizvod', 'Kategorija', 'Kolicina', 'Prodajna_Cena', 'Nabavna_Cena', 'Grad'])

# Vestacko dodavanje duplikata
duplirani_indeksi = np.random.choice(df.index, size=40, replace=False)
df_dupl = df.loc[duplirani_indeksi]
df = pd.concat([df, df_dupl], ignore_index=True)

# Snimanje u CSV
df.to_csv('kompanija_prodaja.csv', index=False)
print("Dataset 'kompanija_prodaja.csv' je uspesno kreiran sa 1240 redova!")

### 1. Ciscenje podataka

In [ ]:
df = pd.read_csv("D:/Python/libraries/kompanija_prodaja.csv")

df.drop_duplicates(keep="first", inplace=True) # 1240 redova pre, 1200 posle, 40 duplikata - Completed!

kolicina_average = df["Kolicina"].median()
df["Kolicina"] = df["Kolicina"].fillna(kolicina_average) # 55 ih bilo nan, sad ih je 55 sa kolicinom "~3"
df["Grad"] = df["Grad"].fillna("Nepoznato") # 185 ih bilo nan, sad ih je 185 sa gradom "Nepoznato"

### 2. Ekstrakcija i transformacija

In [389]:
df["Kategorija"] = df["Kategorija"].astype("category")
df["Kolicina"] = df["Kolicina"].astype(int)
df["Datum"] = pd.to_datetime(df["Datum"], errors="coerce")
df["Prodajna_Cena"] = df["Prodajna_Cena"].str.replace(" RSD", "").str.strip().astype(float)
df["ID_Porudzbine"] = df["ID_Porudzbine"].str.replace("ORD_", "").str.strip().astype(int)

df["Dan Nedelje"] = df["Datum"].dt.weekday
df["Mesec"] = df["Datum"].dt.month
# df["Mesec"] = df["Datum"].dt.strftime('%m - %B')
df["Godina"] = df["Datum"].dt.year

### 3. Analiza i grupisanje

In [390]:
df["Ukupan_Prihod"] = df["Kolicina"] * df["Prodajna_Cena"]
df["Ukupan_Trosak"] = df["Kolicina"] * df["Nabavna_Cena"]
df["Neto_Profit"] = df["Ukupan_Prihod"] - df["Ukupan_Trosak"]

profit_po_gradu = df.groupby("Grad").agg(suma = ("Neto_Profit", "sum")).reset_index().sort_values(by="suma", ascending=False) # Kragujevac
kolicina_po_kategoriji = df.groupby("Kategorija")["Kolicina"].mean() # isto sve

C:\Users\Bobo\AppData\Local\Temp\ipykernel_18572\3124418830.py:6: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



### 4. Plotly

In [391]:
prihod_po_mesecima = df.groupby("Mesec").agg(suma = ("Ukupan_Prihod", "sum")).reset_index()

fig = px.line(prihod_po_mesecima,
              x = "Mesec",
              y = "suma",
              markers=True,
              template="plotly_dark")

fig.update_layout(title=dict(text="Prihod po Mesecima", font=dict(size=18, color="#00C3FF"), xanchor="center", x=0.5))
fig.update_xaxes(title=dict(text="Mesec", font=dict(size=14, color="#00C3FF")), nticks=13)
fig.update_yaxes(title=dict(text="Ukupan Prihod", font=dict(size=14, color="#00C3FF")))
fig.update_traces(line_color="#00C3FF", hovertemplate="<b>Mesec</b>: %{x}<br><b>Prihod</b>: %{y:,.0f}")

fig.show()

In [392]:
fig = px.bar(profit_po_gradu,
              x = "Grad",
              y = "suma",
              color = "suma",
              color_continuous_scale=["#FF2911", "#00FF55"],
              template="plotly_dark")

fig.update_layout(title=dict(text="Prihod po Gradovima", font=dict(size=18, color="#00C3FF"), xanchor="center", x=0.5), coloraxis_showscale=False)
fig.update_xaxes(title=dict(text="Grad", font=dict(size=14, color="#00C3FF")), nticks=13)
fig.update_yaxes(title=dict(text="Ukupan Prihod", font=dict(size=14, color="#00C3FF")))
fig.update_traces(hovertemplate="<b>Mesec</b>: %{x}<br><b>Prihod</b>: %{y:,.0f}")

fig.show()

### 5. Machine Learning

In [393]:
X = df["Proizvod"]
y = df["Kategorija"]

vectorizer = TfidfVectorizer()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

prediction = model.predict(X_test_tfidf)
print(classification_report(y_test, prediction))


              precision    recall  f1-score   support

 Elektronika       1.00      1.00      1.00       117
    Namestaj       1.00      1.00      1.00        49
  Periferije       1.00      1.00      1.00        74

    accuracy                           1.00       240
   macro avg       1.00      1.00      1.00       240
weighted avg       1.00      1.00      1.00       240

